In [ ]:
# IMPORT LIBRARIES
import ee
import geemap
import geopandas as gpd
import pandas as pd
from datetime import datetime

# AUTHENTICATE EARTH ENGINE

ee.Authenticate()

ee.Initialize(
    project='skilful-firefly-507505-q3'
)

print("Earth Engine initialized successfully")

Earth Engine initialized successfully


In [ ]:
# LOAD MATHIRA EAST SUBCOUNTY AOI SHAPEFILE
Mathira_east = "/content/Mathira_East_Subcounty.shp"

Mathira_east_gdf = gpd.read_file(Mathira_east)

print("CRS:", Mathira_east_gdf.crs)
print("Number of features:", len(Mathira_east_gdf))
print(Mathira_east_gdf.head())

CRS: EPSG:4326
Number of features: 1
    fid country provpcode province  ctypcode county   scpcode  \
0  77.0   Kenya   KEN_1_4  Central  KEN_2_19  Nyeri  KEN_3_96   

                 subcounty     dhis2_id        Area  \
0  Mathira East Sub County  vznKK4IegIL  148.262017   

                                            geometry  
0  POLYGON ((37.20401 -0.42653, 37.20407 -0.42658...  


In [ ]:
# VISUALIZING MATHIRA EAST SUBCOUNTY AOI
Map = geemap.Map()

Map.add_basemap("Esri.WorldImagery")

Map.add_gdf(
    Mathira_east_gdf,
    layer_name="Mathira East SubCounty AOI",
    style={
        "color":"red",
        "weight":3,
        "fillOpacity":0
    }
)

Map.centerObject(
    geemap.gdf_to_ee(Mathira_east_gdf),
    12
)

Map

Map(center=[-0.46088155925336455, 37.13895814983191], controls=(WidgetControl(options=['position', 'transparen…

In [ ]:
# REQUIRED SENTINEL-2 BANDS
required_bands = [
    "B1",
    "B2",
    "B3",
    "B4",
    "B5",
    "B6",
    "B7",
    "B8",
    "B8A",
    "B9",
    "B11",
    "B12"
]

print(
    "Required bands:",
    required_bands
)

print(
    "Number of bands:",
    len(required_bands)
)

Required bands: ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12']
Number of bands: 12


In [ ]:
# SENTINEL-2 CLOUD / SHADOW / CIRRUS MASK
def mask_s2_clouds(image):

    scl = image.select("SCL")

    # Cloud shadow
    cloud_shadow = scl.eq(3)

    # Clouds
    clouds = (
        scl.eq(8)
        .Or(scl.eq(9))
        .Or(scl.eq(10))
    )

    # Snow / ice
    snow_ice = scl.eq(11)

    # Combine unwanted pixels
    bad_pixels = (
        cloud_shadow
        .Or(clouds)
        .Or(snow_ice)
    )

    # Keep only valid pixels
    good_mask = bad_pixels.Not()

    return (
        image
        .updateMask(good_mask)
        .copyProperties(
            image,
            image.propertyNames()
        )
    )

In [ ]:
# DRY-SEASON MONTHS
# January
# February
# June
# July
# August
# September

dry_months = [
    1,
    2,
    6,
    7,
    8,
    9
]

print(
    "Dry-season months:",
    dry_months
)

Dry-season months: [1, 2, 6, 7, 8, 9]


In [ ]:
# GET DRY-SEASON SENTINEL-2 IMAGE CANDIDATES

def get_candidate_collection(
    year,
    aoi
):

    # Start and end dates

    start_date = f"{year}-01-01"

    end_date = f"{year + 1}-01-01"

    # Start Sentinel-2 collection

    collection = (
        ee.ImageCollection(
            "COPERNICUS/S2_SR_HARMONIZED"
        )

        # Images must intersect Mathira East
        .filterBounds(aoi)

        # Year
        .filterDate(
            start_date,
            end_date
        )

        # ONLY specified dry-season months

        .filter(
            ee.Filter.Or(
                *[
                    ee.Filter.calendarRange(
                        month,
                        month,
                        "month"
                    )
                    for month in dry_months
                ]
            )
        )
        .filter(
            ee.Filter.lt(
                "CLOUDY_PIXEL_PERCENTAGE",
                10
            )
        )

        # Cloud/shadow masking
        .map(mask_s2_clouds)
    )

    return collection

In [ ]:
# CHECK WHETHER A SCENE FOOTPRINT COMPLETELY COVERS AOI
def image_covers_aoi(
    image,
    aoi
):

    return image.geometry().contains(
        aoi,
        ee.ErrorMargin(1)
    )

In [ ]:
# SELECT BEST FULL-COVERAGE SENTINEL-2 SCENE

def get_best_full_coverage_image(
    year,
    aoi
):

    print(
        f"\nSearching Mathira East imagery for {year}..."
    )

    # Get dry-season candidates

    candidates = get_candidate_collection(
        year,
        aoi
    )

    candidate_count = (
        candidates.size().getInfo()
    )

    print(
        f"Candidate scenes (<10% cloud): "
        f"{candidate_count}"
    )


    # If no candidates exist

    if candidate_count == 0:

        print(
            f"No <10% cloud scenes found for {year}."
        )

        return None
    # Convert collection to list

    candidate_list = candidates.toList(
        candidate_count
    )

    full_coverage_images = []

    # Test every candidate
    for i in range(candidate_count):

        image = ee.Image(
            candidate_list.get(i)
        )

        covers = image_covers_aoi(
            image,
            aoi
        ).getInfo()

        properties = (
            image.getInfo()["properties"]
        )

        cloud_cover = properties.get(
            "CLOUDY_PIXEL_PERCENTAGE",
            None
        )

        tile = properties.get(
            "MGRS_TILE",
            "N/A"
        )

        # Print candidate information

        print(
            f"Candidate {i+1}/{candidate_count} | "
            f"Cloud: {cloud_cover}% | "
            f"Tile: {tile} | "
            f"Full coverage: {covers}"
        )

        # Keep ONLY full-coverage images

        if covers:

            full_coverage_images.append(
                image
            )

    # No full-coverage scene

    if len(full_coverage_images) == 0:

        print(
            f"\nWARNING: No 100% coverage scene "
            f"found for {year}."
        )

        return None
    # Convert full-coverage images to collection
    full_collection = (
        ee.ImageCollection.fromImages(
            full_coverage_images
        )
    )

    # Sort ONLY full-coverage scenes by cloud cover

    best = (
        full_collection
        .sort(
            "CLOUDY_PIXEL_PERCENTAGE"
        )
        .first()
    )

    # Print selected scene

    selected_properties = (
        best.getInfo()["properties"]
    )

    print(
        "\nSELECTED SCENE"
    )

    print(
        "Year:",
        year
    )

    print(
        "Cloud cover:",
        selected_properties.get(
            "CLOUDY_PIXEL_PERCENTAGE",
            "N/A"
        )
    )

    print(
        "Tile:",
        selected_properties.get(
            "MGRS_TILE",
            "N/A"
        )
    )

    return best

In [ ]:
# SELECT FINAL MATHIRA EAST SCENE FOR EACH YEAR

selected_images = {}

metadata = []

for year in range(
    2021,
    2027
):

    print("\n")
    print("=" * 70)
    print(
        f"MATHIRA EAST — YEAR {year}"
    )
    print("=" * 70)

    image = get_best_full_coverage_image(
        year,
        Mathira_east_geom
    )

    # If no suitable scene exists
    if image is None:

        print(
            f"No suitable scene selected for {year}."
        )

        continue
    # Store selected image

    selected_images[year] = image

    # Extract properties
    props = image.getInfo()[
        "properties"
    ]

    acquisition_date = (
        ee.Date(
            image.get(
                "system:time_start"
            )
        )
        .format("YYYY-MM-dd")
        .getInfo()
    )

    # Metadata

    metadata.append({

        "SubCounty":
            "Mathira_East",

        "Year":
            year,

        "Acquisition_Date":
            acquisition_date,

        "CloudCover_%":
            props.get(
                "CLOUDY_PIXEL_PERCENTAGE",
                None
            ),

        "MGRS_Tile":
            props.get(
                "MGRS_TILE",
                "N/A"
            ),

        "Product_ID":
            props.get(
                "PRODUCT_ID",
                "N/A"
            ),

        "Dataset":
            "COPERNICUS/S2_SR_HARMONIZED",

        "Coverage":
            "100%",

        "Cloud_Masking":
            "SCL",

        "Resolution":
            "10 m",

        "Bands":
            ",".join(
                required_bands
            )
    })

# CREATE METADATA DATAFRAME

metadata_df = pd.DataFrame(
    metadata
)

metadata_df



MATHIRA EAST — YEAR 2021

Searching Mathira East imagery for 2021...
Candidate scenes (<10% cloud): 9
Candidate 1/9 | Cloud: 3.987538% | Tile: 37MBV | Full coverage: True
Candidate 2/9 | Cloud: 9.348914% | Tile: 37MCV | Full coverage: False
Candidate 3/9 | Cloud: 1.544922% | Tile: 37MBV | Full coverage: True
Candidate 4/9 | Cloud: 0.456929% | Tile: 37MBV | Full coverage: True
Candidate 5/9 | Cloud: 0.623103% | Tile: 37MCV | Full coverage: False
Candidate 6/9 | Cloud: 1.222279% | Tile: 37MBV | Full coverage: True
Candidate 7/9 | Cloud: 0.000438% | Tile: 37MBV | Full coverage: True
Candidate 8/9 | Cloud: 0.000438% | Tile: 37MCV | Full coverage: False
Candidate 9/9 | Cloud: 8.873133% | Tile: 37MCV | Full coverage: False

SELECTED SCENE
Year: 2021
Cloud cover: 0.000438
Tile: 37MBV


MATHIRA EAST — YEAR 2022

Searching Mathira East imagery for 2022...
Candidate scenes (<10% cloud): 9
Candidate 1/9 | Cloud: 0.349365% | Tile: 37MBV | Full coverage: True
Candidate 2/9 | Cloud: 8.251809% | Ti

Candidate 5/15 | Cloud: 3.476408% | Tile: 37MBV | Full coverage: True
Candidate 6/15 | Cloud: 8.805177% | Tile: 37MBV | Full coverage: True
Candidate 7/15 | Cloud: 0.732848% | Tile: 37MBV | Full coverage: True
Candidate 8/15 | Cloud: 6.814138% | Tile: 37MCV | Full coverage: False
Candidate 9/15 | Cloud: 3.037415% | Tile: 37MBV | Full coverage: True
Candidate 10/15 | Cloud: 8.135192% | Tile: 37MCV | Full coverage: False
Candidate 11/15 | Cloud: 8.219276% | Tile: 37MCV | Full coverage: False
Candidate 12/15 | Cloud: 6.845853% | Tile: 37MCV | Full coverage: False
Candidate 13/15 | Cloud: 4.420924% | Tile: 37MCV | Full coverage: False
Candidate 14/15 | Cloud: 4.530433% | Tile: 37MBV | Full coverage: True
Candidate 15/15 | Cloud: 4.353177% | Tile: 37MCV | Full coverage: False

SELECTED SCENE
Year: 2025
Cloud cover: 0.554836
Tile: 37MBV


MATHIRA EAST — YEAR 2026

Searching Mathira East imagery for 2026...
Candidate scenes (<10% cloud): 13
Candidate 1/13 | Cloud: 4.539341% | Tile: 37MBV | Fu

,SubCounty,Year,Acquisition_Date,CloudCover_%,MGRS_Tile,Product_ID,Dataset,Coverage,Cloud_Masking,Resolution,Bands
0,Mathira_East,2021,2021-02-27,0.000438,37MBV,S2A_MSIL2A_20210227T073851_N0500_R092_T37MBV_2...,COPERNICUS/S2_SR_HARMONIZED,100%,SCL,10 m,"B1,B2,B3,B4,B5,B6,B7,B8,B8A,B9,B11,B12"
1,Mathira_East,2022,2022-01-13,0.002681,37MBV,S2A_MSIL2A_20220113T074301_N0510_R092_T37MBV_2...,COPERNICUS/S2_SR_HARMONIZED,100%,SCL,10 m,"B1,B2,B3,B4,B5,B6,B7,B8,B8A,B9,B11,B12"
2,Mathira_East,2023,2023-01-28,0.000000,37MBV,S2A_MSIL2A_20230128T074151_N0509_R092_T37MBV_2...,COPERNICUS/S2_SR_HARMONIZED,100%,SCL,10 m,"B1,B2,B3,B4,B5,B6,B7,B8,B8A,B9,B11,B12"
3,Mathira_East,2024,2024-09-14,1.616504,37MBV,S2B_MSIL2A_20240914T073609_N0511_R092_T37MBV_2...,COPERNICUS/S2_SR_HARMONIZED,100%,SCL,10 m,"B1,B2,B3,B4,B5,B6,B7,B8,B8A,B9,B11,B12"
4,Mathira_East,2025,2025-01-07,0.554836,37MBV,S2A_MSIL2A_20250107T074311_N0511_R092_T37MBV_2...,COPERNICUS/S2_SR_HARMONIZED,100%,SCL,10 m,"B1,B2,B3,B4,B5,B6,B7,B8,B8A,B9,B11,B12"
5,Mathira_East,2026,2026-08-25,3.233908,37MBV,S2B_MSIL2A_20260825T073609_N0512_R092_T37MBV_2...,COPERNICUS/S2_SR_HARMONIZED,100%,SCL,10 m,"B1,B2,B3,B4,B5,B6,B7,B8,B8A,B9,B11,B12"


In [ ]:
# SELECTED YEARS

print(
    "Years successfully selected:"
)

print(
    sorted(
        selected_images.keys()
    )
)

Years successfully selected:
[2021, 2022, 2023, 2024, 2025, 2026]


In [ ]:
# RGB VISUALIZATION PARAMETERS
rgb_vis = {

    "bands": [
        "B4",
        "B3",
        "B2"
    ],

    "min": 0,

    "max": 3000
}

# MULTI-YEAR VISUAL QUALITY CONTROL

Map = geemap.Map()

Map.centerObject(
    Mathira_east_geom,
    12
)

Map.add_basemap(
    "Esri.WorldImagery"
)


# Add each selected scene

for year in sorted(
    selected_images.keys()
):

    Map.addLayer(

        selected_images[year],

        rgb_vis,

        f"Mathira East {year}",

        False
    )

# Add AOI boundary
Map.addLayer(

    Mathira_east_fc.style(
        color="red",
        fillColor="00000000",
        width=3
    ),

    {},

    "Mathira East Boundary"
)


Map

Map(center=[-0.46088155925336455, 37.13895814983191], controls=(WidgetControl(options=['position', 'transparen…

In [ ]:
# CONVERT AOI TO EE FEATURE COLLECTION (re-creating as it might have been cleared)
Mathira_east_fc = geemap.gdf_to_ee(Mathira_east_gdf)
lari_geom = Mathira_east_fc.geometry()

print("Features:", Mathira_east_fc.size().getInfo())

Features: 1


In [ ]:
# GET CANDIDATE IMAGES FOR 2026
candidates_2026 = get_candidate_collection(2026, lari_geom)
candidate_list_2026 = candidates_2026.toList(candidates_2026.size().getInfo())

# Select the 3rd candidate image (index 2)
candidate_3_2026 = ee.Image(candidate_list_2026.get(2))

# Visualize the selected candidate image
Map_candidate = geemap.Map()
Map_candidate.centerObject(lari_geom, 12)
Map_candidate.add_basemap("Esri.WorldImagery")

Map_candidate.addLayer(
    candidate_3_2026,
    rgb_vis,
    "2026 Candidate 3 (Cloud: 3.76%)"
)

# Add AOI boundary
Map_candidate.addLayer(
    Mathira_east_fc.style(
        color="red",
        fillColor="00000000",
        width=3
    ),
    {},
    "Mathira East Boundary"
)

Map_candidate

Map(center=[-0.46088155925345886, 37.138958149831794], controls=(WidgetControl(options=['position', 'transpare…